# Multivariable Calculus with Vulkan Profiles

This notebook implements the multivariable part of the math requirements: local sensitivities, interaction effects, and a constrained optimization example built on the flattened Vulkan feature set.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from src.utils import fs_utils
from src.vulkan_math_data import (
    attach_catalog_and_benchmarks,
    feature_matrix,
    load_vulkan_feature_frame,
)

project_root, data_dir = fs_utils.get_data_dir()

sns.set_theme(context="notebook", style="whitegrid")
project_root, data_dir

(PosixPath('/opt/pc-hardware-analysis'),
 PosixPath('/opt/pc-hardware-analysis/data'))

In [2]:
vulkan_frame = load_vulkan_feature_frame(data_dir / "vulkan-profiles")
joined_frame, diagnostics = attach_catalog_and_benchmarks(vulkan_frame)
base_matrix, _ = feature_matrix(joined_frame)

candidate_features = [
    "agg__feature_true_count",
    "agg__descriptor_feature_count",
    "agg__raytracing_feature_count",
    "agg__extension_count",
    "agg__format_support_nonempty_count",
]
selected_features = [
    feature for feature in candidate_features if feature in base_matrix.columns
]
X = joined_frame[selected_features].fillna(0)

joined_frame["target_score"] = joined_frame["benchmark"]
target_name = "benchmark"
if joined_frame["target_score"].notna().sum() < 2:
    target_name = "surrogate_value_score"
    joined_frame["target_score"] = (
        joined_frame["agg__feature_true_count"]
        + joined_frame["agg__descriptor_feature_count"]
        + 0.5 * joined_frame["agg__raytracing_feature_count"]
    )

y = joined_frame["target_score"].fillna(joined_frame["target_score"].median())
display(joined_frame[["report_id", "normalized_gpu_name", "benchmark", "target_score"]])
print("Optimization target used for this run:", target_name)

,report_id,normalized_gpu_name,benchmark,target_score
0,49077,rtx 4070 ti,101.0,445.5
1,49118,ryzen 9 7900x 12 core processor,NaN,389.0


Optimization target used for this run: surrogate_value_score


In [3]:
sensitivity_model = Pipeline(
    [
        ("scale", StandardScaler()),
        ("ridge", Ridge(alpha=1.0)),
    ]
)
sensitivity_model.fit(X, y)
coefficients = sensitivity_model.named_steps["ridge"].coef_
sensitivity = pd.DataFrame(
    {
        "feature": selected_features,
        "gradient_on_standardized_scale": coefficients,
        "absolute_effect": np.abs(coefficients),
    }
).sort_values("absolute_effect", ascending=False)
display(sensitivity)

,feature,gradient_on_standardized_scale,absolute_effect
0,agg__feature_true_count,11.3,11.3
1,agg__raytracing_feature_count,11.3,11.3


In [4]:
if len(selected_features) >= 2:
    interaction_model = Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "poly",
                PolynomialFeatures(degree=2, include_bias=False, interaction_only=True),
            ),
            ("ridge", Ridge(alpha=1.0)),
        ]
    )
    interaction_model.fit(X, y)
    interaction_names = interaction_model.named_steps["poly"].get_feature_names_out(
        selected_features
    )
    interaction_coefficients = interaction_model.named_steps["ridge"].coef_
    interaction_frame = pd.DataFrame(
        {
            "term": interaction_names,
            "coefficient": interaction_coefficients,
            "absolute_effect": np.abs(interaction_coefficients),
        }
    )
    interaction_frame = interaction_frame[interaction_frame["term"].str.contains(" ")]
    display(interaction_frame.sort_values("absolute_effect", ascending=False).head(10))
else:
    print(
        "Interaction analysis skipped because fewer than two modeled features are available."
    )

,term,coefficient,absolute_effect
2,agg__feature_true_count agg__raytracing_featur...,0.0,0.0


In [5]:
predicted = sensitivity_model.predict(X)
candidate_table = joined_frame[
    ["report_id", "normalized_gpu_name"] + selected_features
].copy()
candidate_table["predicted_score"] = predicted

descriptor_threshold = (
    candidate_table["agg__descriptor_feature_count"].median()
    if "agg__descriptor_feature_count" in candidate_table
    else 0
)
ray_threshold = (
    candidate_table["agg__raytracing_feature_count"].median()
    if "agg__raytracing_feature_count" in candidate_table
    else 0
)

feasible = candidate_table.copy()
if "agg__descriptor_feature_count" in feasible:
    feasible = feasible[
        feasible["agg__descriptor_feature_count"] >= descriptor_threshold
    ]
if "agg__raytracing_feature_count" in feasible:
    feasible = feasible[feasible["agg__raytracing_feature_count"] >= ray_threshold]

print("Descriptor threshold:", descriptor_threshold)
print("Ray tracing threshold:", ray_threshold)
display(feasible.sort_values("predicted_score", ascending=False))

if not feasible.empty:
    best_candidate = feasible.sort_values("predicted_score", ascending=False).iloc[0]
    print(
        "Best feasible candidate:",
        best_candidate["report_id"],
        "-",
        best_candidate["normalized_gpu_name"],
    )
else:
    print("No feasible candidate satisfied the current constraints.")

Descriptor threshold: 0
Ray tracing threshold: 37.5


,report_id,normalized_gpu_name,agg__feature_true_count,agg__raytracing_feature_count,predicted_score
0,49077,rtx 4070 ti,385,39,439.85


Best feasible candidate: 49077 - rtx 4070 ti
